In [ ]:
import pandas as pd

def extract_target_genes_subset():
    """
    Extract only the target cytokine/chemokine genes from your dataset
    """
    
    # Your exact target genes list
    target_genes = {
        # Main targets with common mouse/human variants
        'TNF': ['TNF', 'Tnf', 'TNF-α'],
        'IL6': ['IL6', 'Il6', 'IL-6'],
        'IL10': ['IL10', 'Il10', 'IL-10'],
        'IFNG': ['IFNG', 'Ifng', 'IFN-γ', 'Ifnγ'],
        'VEGFA': ['VEGFA', 'Vegfa', 'VEGF-A', 'VEGF', 'Vegf'],
        'CSF2': ['CSF2', 'Csf2', 'GM-CSF'],
        'CSF3': ['CSF3', 'Csf3', 'G-CSF'],
        'CCL2': ['CCL2', 'Ccl2', 'MCP-1'],
        'CXCL10': ['CXCL10', 'Cxcl10', 'IP-10'],
        'CCL21': ['CCL21', 'Ccl21', '6CKine'],
        'IL1B': ['IL1B', 'Il1b', 'IL-1β'],
        'IL1A': ['IL1A', 'Il1a', 'IL-1α'],
        'IL2': ['IL2', 'Il2', 'IL-2'],
        'IL4': ['IL4', 'Il4', 'IL-4'],
        'IL5': ['IL5', 'Il5', 'IL-5'],
        'IL13': ['IL13', 'Il13', 'IL-13'],
        'IL17A': ['IL17A', 'Il17a', 'IL-17A'],
        'TGFB1': ['TGFB1', 'Tgfb1', 'TGF-β1'],
        'CCL5': ['CCL5', 'Ccl5', 'RANTES'],
        'CXCL1': ['CXCL1', 'Cxcl1', 'KC'],
        'CD14': ['CD14', 'Cd14'],
        'TREM1': ['TREM1', 'Trem1'],
        'TNFRSF1A': ['TNFRSF1A', 'Tnfrsf1a', 'TNF-RI'],
        'TNFRSF1B': ['TNFRSF1B', 'Tnfrsf1b', 'TNF-RII'],
    }
    
    print("=== Extracting Target Genes Subset ===")
    
    # File paths
    input_file = '/Users/adityaelayavalli/Downloads/normalized_counts_with_symbols_mygene.csv'
    output_file = '/Users/adityaelayavalli/Downloads/cytokine_panel_genes_only.csv'
    
    try:
        # Read the full dataset
        print(f"Reading full dataset: {input_file}")
        df = pd.read_csv(input_file)
        print(f" Loaded {len(df):,} total genes")
        
        # Get all gene symbols
        all_symbols = df['Gene_Symbol'].dropna().astype(str).tolist()
        symbol_set = set(all_symbols)
        
        # Find which target genes are present
        found_symbols = []
        gene_mapping = {}  # Maps found symbol to target gene name
        
        for main_name, variants in target_genes.items():
            for variant in variants:
                if variant in symbol_set:
                    found_symbols.append(variant)
                    gene_mapping[variant] = main_name
                    print(f" Found: {main_name} → {variant}")
                    break  # Only need to find one variant per gene
        
        print(f"\n✓ Found {len(found_symbols)}/{len(target_genes)} target genes")
        
        # Extract rows with target genes
        target_subset = df[df['Gene_Symbol'].isin(found_symbols)].copy()
        
        # Add a column with the standardized gene names
        target_subset['Target_Gene_Name'] = target_subset['Gene_Symbol'].map(gene_mapping)
        
        # Reorder columns to put important info first
        cols = ['Target_Gene_Name', 'Gene_Symbol', 'Genes'] + [col for col in target_subset.columns if col not in ['Target_Gene_Name', 'Gene_Symbol', 'Genes']]
        target_subset = target_subset[cols]
        
        # Sort by target gene name for easier reading
        target_subset = target_subset.sort_values('Target_Gene_Name')
        
        # Save the subset
        target_subset.to_csv(output_file, index=False)
        
        print(f"\n=== Results ===")
        print(f" Extracted {len(target_subset)} target genes")
        print(f" Saved to: {output_file}")
        
        # Show summary of what was extracted
        print(f"\n=== Extracted Genes Summary ===")
        for _, row in target_subset.iterrows():
            print(f"{row['Target_Gene_Name']}: {row['Gene_Symbol']} ({row['Genes']})")
        
        # Show expression data preview
        print(f"\n=== Expression Data Preview ===")
        expr_cols = [col for col in target_subset.columns if col.startswith('X527_')]
        sample_cols = expr_cols[:3]  # First 3 samples
        
        print(f"Showing first 3 samples: {sample_cols}")
        for _, row in target_subset.head(10).iterrows():  # First 10 genes
            values = [f"{row[col]:.1f}" for col in sample_cols]
            print(f"{row['Target_Gene_Name']} ({row['Gene_Symbol']}): {' | '.join(values)}")
        
        if len(target_subset) > 10:
            print(f"... and {len(target_subset) - 10} more genes")
        
        # Create a summary statistics file
        summary_file = '/Users/adityaelayavalli/Downloads/cytokine_panel_summary.csv'
        
        # Calculate basic stats for each gene
        summary_data = []
        for _, row in target_subset.iterrows():
            expr_values = row[expr_cols].values
            summary_data.append({
                'Target_Gene_Name': row['Target_Gene_Name'],
                'Gene_Symbol': row['Gene_Symbol'],
                'Ensembl_ID': row['Genes'],
                'Mean_Expression': expr_values.mean(),
                'Median_Expression': pd.Series(expr_values).median(),
                'Max_Expression': expr_values.max(),
                'Min_Expression': expr_values.min(),
                'Std_Expression': expr_values.std()
            })
        
        summary_df = pd.DataFrame(summary_data)
        summary_df = summary_df.sort_values('Mean_Expression', ascending=False)
        summary_df.to_csv(summary_file, index=False)
        
        print(f"\n Also created summary statistics: {summary_file}")
        print(f"\nTop 5 highest expressed target genes:")
        for _, row in summary_df.head().iterrows():
            print(f"  {row['Target_Gene_Name']} ({row['Gene_Symbol']}): {row['Mean_Expression']:.1f} (mean)")
        
        return target_subset, summary_df
        
    except Exception as e:
        print(f"✗ Error: {e}")
        return None, None

def create_analysis_ready_file():
    """
    Create a file formatted for easy analysis with proper gene names
    """
    print(f"\n=== Creating Analysis-Ready File ===")
    
    input_file = '/Users/adityaelayavalli/Downloads/cytokine_panel_genes_only.csv'
    output_file = '/Users/adityaelayavalli/Downloads/cytokine_panel_for_analysis.csv'
    
    try:
        df = pd.read_csv(input_file)
        
        # Create a clean version with standardized names
        analysis_df = df.copy()
        
        # Use Target_Gene_Name as the primary identifier
        analysis_df = analysis_df.drop(['Gene_Symbol', 'Genes'], axis=1)
        
        # Set Target_Gene_Name as index for easier analysis
        analysis_df = analysis_df.set_index('Target_Gene_Name')
        
        # Save
        analysis_df.to_csv(output_file)
        
        print(f" Created analysis-ready file: {output_file}")
        print(f" Genes are now indexed by standardized names (TNF, IL6, etc.)")
        print(f" Ready for heatmaps, clustering, differential expression analysis")
        
        return analysis_df
        
    except Exception as e:
        print(f" Error creating analysis file: {e}")
        return None

if __name__ == "__main__":
    # Extract the target genes
    subset_df, summary_df = extract_target_genes_subset()
    
    if subset_df is not None:
        # Create analysis-ready version
        analysis_df = create_analysis_ready_file()
        
        print(f" SUCCESS! Created 3 files:")
        print(f"1. cytokine_panel_genes_only.csv - Full data with all columns")
        print(f"2. cytokine_panel_summary.csv - Summary statistics")
        print(f"3. cytokine_panel_for_analysis.csv - Clean format for analysis")
    else:
        print("Failed to extract target genes.")